# YaSpeech: автопротоколирование деловых встреч на Yandex Cloud
### SpeechKit (ASR) + YandexGPT + Object Storage

Кукбук показывает, как собрать сервис **«запись встречи → готовый протокол»** на трёх сервисах Yandex Cloud.


# 1. Введение

## Назначение

В этом кукбуке демонстрируется, как собрать сервис **«запись встречи → готовый протокол»** с помощью:
- **SpeechKit STT v3** — асинхронное распознавание речи с разметкой по каналам
- **YandexGPT** — коррекция ошибок ASR, генерация протокола
- **Object Storage** — промежуточное хранение аудио (обязательно: SpeechKit async читает файл только по ссылке)

## Описание задачи

**Бизнес-задача**: команды, которые фиксируют решения и задачи по итогам планёрок, тратят время на ручное протоколирование. Этот кукбук демонстрирует:

- Распознавание речи из аудиозаписи встречи
- Коррекцию типичных ошибок ASR (разорванные слова, аббревиатуры, произнесённые словами числа)
- Генерацию структурированного протокола: решения, задачи с ответственными, открытые вопросы

**Основные сервисы Yandex Cloud:**
- SpeechKit STT v3
- AI Studio с YandexGPT
- Object Storage

***

## Ожидаемый результат

Готовый пайплайн «аудио → протокол» — рабочий каркас, от которого можно оттолкнуться и построить решение под свою задачу.


# 2. Архитектура решения

## Компоненты системы

```
Аудио (.wav/.ogg/.mp3)
      │
      ▼
Object Storage (SpeechKit читает аудио только по ссылке)
      │
      ▼
SpeechKit STT v3 — асинхронное распознавание
      │
      ▼
YandexGPT:
  короткая встреча — один вызов (коррекция ASR + протокол)
  длинная встреча — чанкинг: коррекция по кускам, протокол по всему тексту
      │
      ▼
Структурированный протокол (JSON)
```

| Компонент | Назначение | Роли и права |
|---|---|---|
| **SpeechKit STT v3** | Распознавание речи (`recognizeFileAsync` + `getRecognition`) | `ai.speechkit-stt.user` — вызов распознавания |
| **YandexGPT** | Коррекция ошибок ASR, протокол — один вызов для коротких встреч, чанкинг для длинных (см. раздел 6) | `ai.languageModels.user` — генерация текста |
| **Object Storage** | Промежуточное хранение аудио для SpeechKit | `storage.editor` — создание бакета и загрузка объектов |

## Описание ролей и их области действия

**Сервисный аккаунт** — учётная запись, от имени которой приложения и автоматизированные сервисы обращаются к ресурсам Yandex Cloud. В отличие от обычных пользовательских аккаунтов, сервисные аккаунты используются для программного доступа и не требуют браузерной аутентификации.

- `ai.speechkit-stt.user` — *область действия: каталог и вложенные ресурсы.* Позволяет отправлять запросы на распознавание речи в SpeechKit (`recognizeFileAsync`, `getRecognition`).
- `ai.languageModels.user` — *область действия: каталог и вложенные ресурсы.* Минимальная роль для работы с моделями генерации текста YandexGPT — отправка запросов на генерацию.
- `storage.editor` — *область действия: каталог, вложенные ресурсы, либо конкретный бакет.* Даёт право создавать бакет и загружать в него объекты — обе операции нужны коду кукбука (`ensure_bucket`, `upload_audio`). Более узкая роль `storage.uploader` разрешает только загрузку в уже существующий бакет, но не его создание — если бакет уже создан заранее, можно использовать её вместо `storage.editor`.

## Как назначить роль сервисному аккаунту через консоль Yandex Cloud

1. Откройте сервис Identity and Access Management (IAM) в консоли управления.
2. Выберите каталог, к которому нужно предоставить доступ.
3. Перейдите на вкладку «Права доступа» → «Настроить доступ».
4. Выберите раздел «Сервисные аккаунты», найдите нужный или создайте новый.
5. Нажмите «Добавить роль» и выберите роли из списка выше — можно назначить несколько ролей сразу.
6. Сохраните изменения.


# 3. Подготовка окружения

Понадобятся три вещи:
- **FOLDER_ID** — идентификатор каталога: https://yandex.cloud/ru/docs/resource-manager/operations/folder/get-id
- **YC_API_KEY** — ключ сервисного аккаунта с ролями `ai.speechkit-stt.user`, `ai.languageModels.user`, `storage.editor` (см. раздел 2): https://yandex.cloud/ru/docs/iam/operations/api-key/create
- **AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY** — статические ключи для Object Storage: https://yandex.cloud/ru/docs/iam/concepts/authorization/access-key


Устанавливаем нужные библиотеки.


In [ ]:
!pip install -q openai python-dotenv boto3 requests pydantic


In [ ]:
import os
import json
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import boto3
import requests
import openai
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field, field_validator

load_dotenv(find_dotenv())

FOLDER_ID = os.getenv("FOLDER_ID", "YOUR_FOLDER_ID")
YC_API_KEY = os.getenv("YC_API_KEY", "YOUR_API_KEY")
GPT_MODEL = os.getenv("GPT_MODEL", "yandexgpt-5-lite")  # см. актуальный список моделей: https://yandex.cloud/ru/docs/ai-studio/concepts/generation/models
MODEL_URI = f"gpt://{FOLDER_ID}/{GPT_MODEL}"

gpt_client = openai.OpenAI(
    api_key=YC_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
)

S3_BUCKET = os.getenv("BUCKET_NAME", "YOUR_BUCKET_NAME")
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "YOUR_AWS_ACCESS_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "YOUR_AWS_SECRET_KEY"),
    region_name="ru-central1",
)

SPEECHKIT_RECOGNIZE_URL = "https://stt.api.cloud.yandex.net/stt/v3/recognizeFileAsync"
SPEECHKIT_GET_URL = "https://stt.api.cloud.yandex.net/stt/v3/getRecognition"
OPERATION_URL = "https://operation.api.cloud.yandex.net/operations"

print("Клиенты инициализированы:", MODEL_URI)


# 4. Загрузка аудио в Object Storage

SpeechKit async принимает аудио **только по ссылке** на Object Storage — поэтому bucket и upload обязательны, даже в учебном примере.

Поддерживаемые контейнеры: **WAV, OggOpus, MP3** (без M4A).


In [ ]:
def ensure_bucket(bucket_name: str) -> None:
    """Проверяет, есть ли уже такой бакет, и создаёт его, если нет."""
    existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
    if bucket_name not in existing:
        s3.create_bucket(Bucket=bucket_name)
    print(f"Бакет готов: {bucket_name}")


def upload_audio(local_path: str, key: str) -> str:
    """Загружает локальный аудиофайл в Object Storage и возвращает
    публичную ссылку на него — эту ссылку потом передаём в SpeechKit."""
    s3.upload_file(local_path, S3_BUCKET, key)
    uri = f"https://storage.yandexcloud.net/{S3_BUCKET}/{key}"
    print(f"Загружено: {uri}")
    return uri


ensure_bucket(S3_BUCKET)


# 5. SpeechKit STT v3: асинхронное распознавание

Отправляем аудио на распознавание — SpeechKit сразу отвечает номером операции и продолжает работу в фоне. Мы периодически спрашиваем «готово?», а когда готово — забираем результат: текст, разбитый на реплики.

У SpeechKit есть встроенная функция `speakerLabeling` — она пытается сама определить, кто где говорит, но работает только для одноканальной записи и максимум на двух дикторов, и на практике часто ошибается: если собеседники говорят в один микрофон, границы между репликами получаются смазанными. Это скорее разметка по акустике и паузам, чем настоящая диаризация — она не понимает, кто есть кто, только угадывает смену голоса. Мы её не включаем.

Более того, на записи, сведённой в стерео с одного микрофона, SpeechKit нередко распознаёт один и тот же звук на обоих аудиоканалах и присылает два почти идентичных набора реплик под разными `channelTag` — это не два говорящих, а дублирование одного и того же звука. Поэтому мы берём текст только с одного канала (`channelTag == "0"`) и подписываем каждую реплику отметкой времени её начала (`[MM:SS]`) вместо номера канала — так видно, когда что было сказано, без ложного разделения на "спикеров", которых на самом деле нет. Если вам нужны реальные имена участников, для этого нужна акустическая диаризация от специализированного сервиса — этот кукбук её не показывает.

Скорость примерно такая: минута записи распознаётся около 10 секунд ([подробнее в документации](https://aistudio.yandex.ru/docs/ru/speechkit/stt/api/transcribation-api-v3)).


In [ ]:
def start_recognition(audio_uri: str, language: str = "ru-RU") -> str:
    """Запускает асинхронное распознавание по ссылке на файл в Object Storage
    и сразу возвращает id операции — сам результат появится позже."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}", "Content-Type": "application/json"}
    body = {
        "uri": audio_uri,
        "recognitionModel": {
            "model": "general",
            "audioFormat": {"containerAudio": {"containerAudioType": "WAV"}},
            "textNormalization": {
                "textNormalization": "TEXT_NORMALIZATION_ENABLED",
                "profanityFilter": False,
                "literatureText": True,
            },
            "languageRestriction": {"restrictionType": "WHITELIST", "languageCode": [language]},
        },
        # speakerLabeling нарочно не включаем — она угадывает смену голоса по
        # акустике, а не понимает, кто есть кто, и ненадёжна при более чем
        # двух участниках (см. раздел 5).
    }
    resp = requests.post(SPEECHKIT_RECOGNIZE_URL, headers=headers, json=body, timeout=30)
    resp.raise_for_status()
    operation_id = resp.json()["id"]
    print(f"Операция запущена: {operation_id}")
    return operation_id


def wait_operation(operation_id: str, poll_interval_s: int = 5, max_wait_s: int = 900) -> None:
    """Опрашивает статус операции распознавания, пока она не завершится
    (done: true), или бросает исключение по таймауту/ошибке SpeechKit."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    waited = 0
    while waited < max_wait_s:
        resp = requests.get(f"{OPERATION_URL}/{operation_id}", headers=headers, timeout=15)
        resp.raise_for_status()
        op = resp.json()
        if op.get("done"):
            if op.get("error"):
                raise RuntimeError(f"SpeechKit error: {op['error']}")
            return
        time.sleep(poll_interval_s)
        waited += poll_interval_s
    raise TimeoutError("SpeechKit: не завершилось за отведённое время")


def fetch_recognition(operation_id: str) -> List[Dict[str, Any]]:
    """Забирает результат завершённой операции — ответ приходит построчным
    JSON (NDJSON), функция разбирает его в список {start_ms, text}.

    Берём только channelTag == "0": на записи с одного микрофона, сведённой в
    стерео, SpeechKit нередко распознаёт один и тот же звук на обоих каналах
    и отдаёт два почти идентичных набора реплик — второй канал в таком случае
    просто дублирует первый, а не несёт отдельного голоса. Канал 0 всегда
    присутствует, поэтому его достаточно как единственного источника текста."""
    headers = {"Authorization": f"Api-Key {YC_API_KEY}"}
    resp = requests.get(SPEECHKIT_GET_URL, headers=headers,
                        params={"operationId": operation_id}, timeout=60)
    resp.raise_for_status()

    chunks = []
    for line in resp.text.splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        final = obj.get("result", {}).get("final")
        if not final:
            continue
        if final.get("channelTag", "0") != "0":
            continue
        alternatives = final.get("alternatives", [])
        if not alternatives or not alternatives[0].get("text", "").strip():
            continue
        start_ms = int(alternatives[0].get("startTimeMs", 0))
        end_ms = int(alternatives[0].get("endTimeMs", start_ms))
        chunks.append({"start_ms": start_ms, "end_ms": end_ms, "text": alternatives[0]["text"].strip()})
    return chunks


def recognize_meeting_audio(audio_uri: str) -> List[Dict[str, Any]]:
    """Полный цикл распознавания одного файла: запустить, дождаться,
    забрать результат."""
    operation_id = start_recognition(audio_uri)
    wait_operation(operation_id)
    return fetch_recognition(operation_id)


## 5.1 Постобработка транскрипта

Фильтруем короткие слова-паразиты и склеиваем реплики, между которыми пауза короче 8 секунд — SpeechKit режет по паузам в звуке, а не по смыслу, и без склейки одно предложение превращается в россыпь коротких строк. Результат — список сегментов с таймкодами, а не готовый текст: метки времени в текст не встраиваются здесь и не будут встраиваться до самого конца пайплайна (см. раздел 6, почему).


In [ ]:
NOISE_MARKERS = {"угу", "ага", "эм", "эээ", "ну", "вот", "это", "так"}


def is_noise_segment(text: str) -> bool:
    """Короткая реплика из одних слов-паразитов («угу», «ну» и т.п.) —
    такие сегменты не несут смысла, их отфильтровываем."""
    words = text.lower().split()
    return bool(words) and len(words) <= 3 and all(w in NOISE_MARKERS for w in words)


def format_timestamp(start_ms: int) -> str:
    """MM:SS от начала записи."""
    total_seconds = start_ms // 1000
    return f"{total_seconds // 60:02d}:{total_seconds % 60:02d}"


MAX_PAUSE_MS_TO_MERGE = 8000  # пауза короче — считаем продолжением той же реплики


def postprocess_transcript(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Фильтрует мусор и склеивает реплики, разделённые короткой паузой (SpeechKit
    режет по паузам в звуке, а не по смыслу — без склейки одно предложение
    разбивается на несколько строк). Возвращает список сегментов {start_ms, end_ms,
    text} — НЕ строку с готовыми метками [MM:SS]. Таймкод остаётся программным
    полем каждого сегмента до самого конца пайплайна: если встроить метку прямо
    в текст, LLM-коррекция ниже по цепочке должна будет сама пронести этот текстовый
    токен через переписывание — а инструкции "не трогай метки" ненадёжны (на
    практике модель на части входов теряет почти все встроенные метки). Программная
    привязка таймкода к тексту исключает этот отказ полностью."""
    filtered = [c for c in chunks if c["text"] and not is_noise_segment(c["text"])]

    merged: List[Dict[str, Any]] = []
    for c in filtered:
        if merged and c["start_ms"] - merged[-1]["end_ms"] <= MAX_PAUSE_MS_TO_MERGE:
            merged[-1]["text"] += " " + c["text"]
            merged[-1]["end_ms"] = c["end_ms"]
        else:
            merged.append(dict(c))

    return merged


# 6. YandexGPT: коррекция и протокол

Коррекция ошибок ASR и извлечение протокола — теперь два независимых шага с разными данными на входе:

1. **Коррекция сегментов** (`correct_all_segments`) — batch-вызовы над JSON-массивами голого текста (без таймкодов). Каждый сегмент несёт свой `start_ms`/`end_ms` в питоновской структуре, и они никогда не передаются модели — значит их и нечего "потерять" при переписывании. Раньше метки `[MM:SS]` встраивались прямо в текст с инструкцией "не трогай метки" — на части входов модель эту инструкцию всё равно нарушала и стирала почти все метки. Это классическая ненадёжность LLM при попытке дословно пронести токен через переписывание текста; надёжное решение — не давать модели вообще видеть то, что должно остаться неизменным.
2. **Извлечение протокола** (`extract_meeting_protocol`) — отдельный вызов над уже исправленным чистым текстом (`build_transcript_text`, без меток времени — они здесь не нужны).

Две инструкции, которые здесь особенно важны:
- **не выдумывай** — пустой список лучше вымышленного решения или задачи;
- **не меняй числа, даты и суммы** при исправлении ошибок распознавания.

Длинный транскрипт не помещается в один вызов коррекции: `max_output_tokens` ограничивает объём генерации. Поэтому сегменты режутся на батчи по суммарной длине текста (`batch_segments`) — это чистая Python-логика без обращения к LLM, в отличие от прежнего подхода с LLM-разметкой смысловых границ: раз батчи не делятся контекстом друг с другом, точная граница между ними не так важна, как раньше, когда неточная граница резала посреди фразы, ключевой для соседнего чанка.


In [ ]:
class ActionItem(BaseModel):
    owner: str = Field(description="Ответственный, или 'Команда' если не назван")
    task: str
    deadline: Optional[str] = None

    @field_validator("deadline", mode="before")
    @classmethod
    def _normalize_null_string(cls, v):
        """Промпт просит null для отсутствующего дедлайна (см. build_protocol_prompt,
        пример deadline: "2026-08-15 или null"), но модель иногда буквально
        подставляет строку "null" вместо JSON null — нормализуем сюда, а не
        оставляем на совесть промпт-инструкции."""
        return None if v == "null" else v


class ProtocolOutput(BaseModel):
    """Pydantic-модель не диктует формат ответа модели (Responses API это
    делает текстовым промптом, см. build_protocol_prompt), а валидирует то,
    что пришло — сразу видно, если модель что-то упустила или исказила тип.

    Транскрипта здесь нет: он собирается отдельно, программно, из уже
    исправленных сегментов с их исходными таймкодами (см. раздел 6.1) — LLM
    в этом протокольном вызове видит только чистый текст без меток времени и
    не должен ничего "проносить" через переписывание."""
    meeting_title: str
    domain: str = Field(description="Предметная сфера встречи, например: строительство, IT, продажи")
    summary: str = Field(description="4-6 предложений деловой прозы")
    decisions: List[str] = Field(default_factory=list, description="Только реально принятые решения, дословно из транскрипта")
    action_items: List[ActionItem] = Field(default_factory=list)
    open_questions: List[str] = Field(default_factory=list, description="Вопросы, которые обсуждали, но не решили")

    @field_validator("meeting_title", mode="before")
    @classmethod
    def _fallback_title(cls, v):
        """Промпт просит модель никогда не оставлять meeting_title пустым (см.
        build_protocol_prompt, п.4), но на коротких/малоинформативных встречах
        модель иногда всё же возвращает null, применяя к заголовку правило
        "не выдумывай" — программная страховка на этот случай, а не только
        промпт-инструкция."""
        return v if v else "Встреча без определённой темы"


class SegmentBatchOutput(BaseModel):
    """Ответ на коррекцию батча сегментов — список исправленных текстов в том
    же порядке и того же размера, что и вход. Никаких таймкодов в этом
    JSON нет и не может быть: таймкод каждого сегмента остаётся в питоновской
    структуре (см. correct_segments_batch), а не в тексте, который отвечает LLM."""
    corrected_texts: List[str] = Field(description="Исправленные тексты сегментов, по одному на каждый входной сегмент, в том же порядке")


In [ ]:
BATCH_SIZE_CHARS = 6000  # суммарная длина сегментов в одном батче коррекции


def build_segment_batch_prompt() -> str:
    """Промпт для коррекции батча сегментов. На вход и на выход — JSON-массив
    текстов, без единого таймкода: таймкод каждого сегмента хранится в питоновской
    структуре снаружи этого вызова, LLM его никогда не видит и не должна ничего
    "проносить" через переписывание — раньше метки [MM:SS] встраивались прямо в
    текст, и модель на части входов теряла почти все встроенные метки, несмотря
    на прямой запрет их трогать. Явных числовых индексов пар вход→выход тоже нет:
    порядок и длина списка — единственный контракт, за него отвечает Pydantic
    (SegmentBatchOutput.corrected_texts должен быть того же размера)."""
    return """Ты — редактор транскриптов деловых встреч. Тебе дан JSON-массив
текстовых фрагментов, полученных автоматическим распознаванием речи (ASR).
Каждый фрагмент — самостоятельный кусок, порядок в массиве важен, но
контекста между фрагментами нет.

ЗАДАЧА для каждого фрагмента:
1. Исправь искажения ASR: разорванные слова, аббревиатуры и марки, произнесённые
   словами ("м триста пятьдесят" → "М350"). НИКОГДА не меняй сами числа, даты и суммы.
2. Пунктуация и заглавные буквы — по смыслу.
3. НЕ объединяй фрагменты между собой и не меняй их количество — на выходе
   должно быть ровно столько же строк текста, сколько на входе, в том же порядке.

Верни ТОЛЬКО корректный JSON без markdown-обёртки:
{"corrected_texts": ["исправленный фрагмент 1", "исправленный фрагмент 2", ...]}"""


def correct_segments_batch(segments: List[Dict[str, Any]]) -> List[str]:
    """Исправляет ошибки ASR в батче сегментов одним вызовом LLM, отдавая и
    принимая только голые тексты (без таймкодов) — таймкод каждого сегмента
    остаётся в вызывающем коде и присоединяется к исправленному тексту по
    индексу списка, а не парсится из ответа модели. Если ответ не прошёл
    валидацию (сеть, невалидный JSON, несовпадение длины списка) — возвращает
    исходные тексты без коррекции, вместо того чтобы ронять весь пайплайн
    из-за одного батча."""
    texts = [s["text"] for s in segments]
    try:
        response = gpt_client.responses.create(
            model=MODEL_URI,
            input=[
                {"role": "system", "content": build_segment_batch_prompt()},
                {"role": "user", "content": json.dumps(texts, ensure_ascii=False)},
            ],
            temperature=0.2,
            max_output_tokens=8000,
        )
        candidate = response.output_text.strip()
        if candidate.startswith("```"):
            candidate = candidate.split("```")[1]
            candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
        corrected = SegmentBatchOutput.model_validate_json(candidate).corrected_texts
        if len(corrected) != len(texts):
            raise ValueError(f"ожидалось {len(texts)} текстов в ответе, получено {len(corrected)}")
        return corrected
    except Exception as e:
        print(f"⚠️  Не удалось исправить батч сегментов ({e}), оставляю как есть")
        return texts


def _split_oversized_segment(segment: Dict[str, Any], max_len: int) -> List[Dict[str, Any]]:
    """Дробит ОДИН сегмент, чей текст сам по себе длиннее max_len, на несколько
    под-сегментов по границам слов. Такое бывает, когда MAX_PAUSE_MS_TO_MERGE
    склеивает всю запись (или её большую часть) в один сегмент, потому что в
    ней просто нет пауз длиннее порога — без этого дробления такой сегмент
    ушёл бы в LLM одним куском, упёрся в max_output_tokens, и весь текст
    остался бы без коррекции ASR (реальный случай: 10591-символьная запись
    без длинных пауз схлопнулась в один сегмент, ответ модели оборвался).
    Все под-сегменты наследуют start_ms/end_ms родителя — деление внутри
    одного склеенного блока не даёт более точных таймкодов, чем у родителя,
    и это не проблема: PARAGRAPH_GAP_MS всё равно не создаст здесь лишних
    меток времени, так как пауз между под-сегментами нет вообще."""
    words = segment["text"].split(" ")
    parts: List[str] = []
    current_words: List[str] = []
    current_len = 0
    for word in words:
        if current_words and current_len + len(word) + 1 > max_len:
            parts.append(" ".join(current_words))
            current_words, current_len = [], 0
        current_words.append(word)
        current_len += len(word) + 1
    if current_words:
        parts.append(" ".join(current_words))

    return [{**segment, "text": part} for part in parts]


def batch_segments(segments: List[Dict[str, Any]], batch_size_chars: int = BATCH_SIZE_CHARS) -> List[List[Dict[str, Any]]]:
    """Группирует сегменты в батчи по суммарной длине текста — простая резка
    по накопленному размеру, без обращения к LLM (в отличие от прежнего подхода
    с LLM-разметкой смысловых границ по номерам строк текста). Программная
    резка надёжнее здесь: каждый батч отправляется на коррекцию независимо
    (без общего контекста, см. correct_segments_batch), поэтому смысловая
    точность границы между батчами не критична.

    Отдельно дробит сегменты, чей текст сам по себе длиннее batch_size_chars
    (см. _split_oversized_segment) — без этого один такой сегмент ушёл бы в
    LLM целиком в одном батче, независимо от лимита."""
    expanded: List[Dict[str, Any]] = []
    for seg in segments:
        if len(seg["text"]) > batch_size_chars:
            expanded.extend(_split_oversized_segment(seg, batch_size_chars))
        else:
            expanded.append(seg)

    batches: List[List[Dict[str, Any]]] = []
    current: List[Dict[str, Any]] = []
    current_len = 0
    for seg in expanded:
        if current and current_len + len(seg["text"]) > batch_size_chars:
            batches.append(current)
            current, current_len = [], 0
        current.append(seg)
        current_len += len(seg["text"])
    if current:
        batches.append(current)
    return batches


In [ ]:
def build_protocol_prompt() -> str:
    """В промпт кладём готовый ПРИМЕР ответа с реальными значениями, а не дамп
    JSON Schema (properties/type/description) — модель на практике путает такую
    схему с данными и подставляет метаданные полей вместо самих значений.

    Здесь нет ни таймкодов, ни просьбы вернуть транскрипт — этот вызов только
    извлекает протокол из уже исправленного текста (см. build_segment_batch_prompt
    для коррекции ASR). Разметку по именам сюда не кладём: у SpeechKit нет
    надёжной акустической диаризации на запись с более чем двумя участниками
    (см. раздел 5), а угадывать имена по одному тексту без голоса ненадёжно."""
    example = {
        "meeting_title": "Планёрка по объекту на Садовой",
        "domain": "строительство",
        "summary": "4-6 предложений деловой прозы.",
        "decisions": ["решение дословно из транскрипта"],
        "action_items": [{"owner": "Команда", "task": "конкретная задача", "deadline": "2026-08-15 или null"}],
        "open_questions": ["вопрос, который обсуждали, но не решили"],
    }
    example_json = json.dumps(example, ensure_ascii=False, indent=2)

    return f"""Ты — секретарь деловых встреч. Тебе дан исправленный транскрипт встречи.
Реплики не разделены по говорящим — не пытайся угадывать по тексту, кто есть кто,
и не выдумывай имена или метки спикеров.

ЗАДАЧА — составь протокол по транскрипту:
1. Реальные решения ("решили", "утвердили"), задачи с ответственным (если из
   текста не ясно, кто именно — пиши "Команда") и дедлайном (если не назван —
   null), открытые вопросы, которые обсуждали, но не решили.
2. "meeting_title" — ВСЕГДА строка, никогда null: если из транскрипта не ясна
   конкретная тема встречи, используй общее описание по домену
   (например "Планёрка по текущим задачам"), а не оставляй поле пустым.

НЕ ВЫДУМЫВАЙ факты. Если решений или задач нет в тексте — верни пустой список,
а не пример "для порядка". Это правило не относится к meeting_title (см. п.2) —
заголовок обязателен как ярлык, даже когда тема расплывчата.

Верни ТОЛЬКО корректный JSON без markdown-обёртки, СТРОГО в этом формате
(ниже — форма ответа с примерами значений, не переписывай примеры дословно):
{example_json}"""


In [ ]:
def parse_protocol_json(raw_text: str) -> ProtocolOutput:
    """Модель иногда оборачивает JSON в ```json ... ``` — снимаем обёртку перед парсингом."""
    candidate = raw_text.strip()
    if candidate.startswith("```"):
        candidate = candidate.split("```")[1]
        candidate = candidate[4:].strip() if candidate.lower().startswith("json") else candidate.strip()
    return ProtocolOutput.model_validate_json(candidate)


def correct_all_segments(segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Исправляет ошибки ASR во всех сегментах, батчами (см. batch_segments,
    correct_segments_batch), и возвращает НОВЫЙ список сегментов той же формы
    {start_ms, end_ms, text}, но с исправленным text — start_ms/end_ms не
    проходят через LLM ни разу, они просто копируются из исходных сегментов
    по индексу. Это гарантирует, что таймкод никогда не теряется, независимо
    от того, что вернула модель."""
    batches = batch_segments(segments)
    if len(batches) > 1:
        print(f"Транскрипт длинный — режем на {len(batches)} батчей коррекции")

    corrected_segments: List[Dict[str, Any]] = []
    for i, batch in enumerate(batches, start=1):
        corrected_texts = correct_segments_batch(batch)
        for seg, corrected_text in zip(batch, corrected_texts):
            corrected_segments.append({**seg, "text": corrected_text})
        if len(batches) > 1:
            print(f"  батч {i}/{len(batches)} исправлен")

    return corrected_segments


PARAGRAPH_GAP_MS = 20000  # пауза длиннее — новый абзац с новой меткой времени


def build_transcript_text(segments: List[Dict[str, Any]]) -> str:
    """Собирает финальный текст транскрипта из уже исправленных сегментов —
    чистая программная сборка, без единого обращения к LLM. Метка [MM:SS]
    ставится только в начале нового абзаца (пауза от предыдущего сегмента
    длиннее PARAGRAPH_GAP_MS), а не перед каждым сегментом — так текст читается
    как связная проза с редкими таймкодами, и при этом каждая метка гарантированно
    верна: она берётся из исходного start_ms сегмента, а не восстанавливается
    моделью из текста."""
    if not segments:
        return ""

    paragraphs: List[str] = []
    current_texts = [segments[0]["text"]]
    current_start = segments[0]["start_ms"]
    prev_end = segments[0]["end_ms"]

    for seg in segments[1:]:
        if seg["start_ms"] - prev_end > PARAGRAPH_GAP_MS:
            paragraphs.append(f"[{format_timestamp(current_start)}] {' '.join(current_texts)}")
            current_texts = []
            current_start = seg["start_ms"]
        current_texts.append(seg["text"])
        prev_end = seg["end_ms"]

    paragraphs.append(f"[{format_timestamp(current_start)}] {' '.join(current_texts)}")
    return "\n\n".join(paragraphs)


def extract_meeting_protocol(plain_text: str) -> ProtocolOutput:
    """Извлекает протокол из чистого текста встречи (без таймкодов — они не
    нужны для решений/задач/вопросов). Текст обрезается до 20000 символов —
    для очень длинных встреч это ограничение учебной версии (см. раздел 9)."""
    system_prompt = build_protocol_prompt()

    try:
        response = gpt_client.responses.create(
            model=MODEL_URI,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Транскрипт:\n{plain_text[:20000]}"},
            ],
            temperature=0.2,
            max_output_tokens=4000,
        )
        return parse_protocol_json(response.output_text)
    except Exception as e:
        # Не глушим ошибку — протокол терять молча нельзя, но печатаем
        # понятную причину вместо голого traceback валидации pydantic.
        print(f"❌ Не удалось собрать протокол: {e}")
        raise


# 7. Сборка пайплайна


In [ ]:
def run_meeting_pipeline(audio_local_path: str) -> Tuple[str, ProtocolOutput, str]:
    """Возвращает (транскрипт, протокол, meeting_id) — id нужен, чтобы сохранить
    результат прогона в файл с тем же именем, что и аудио в Object Storage.
    Транскрипт собирается программно из исправленных сегментов (build_transcript_text)
    и не зависит от того, сохранила ли LLM таймкоды в тексте — она их никогда не видит."""
    meeting_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    audio_uri = upload_audio(audio_local_path, f"audio/{meeting_id}.wav")

    raw_chunks = recognize_meeting_audio(audio_uri)
    segments = postprocess_transcript(raw_chunks)
    print(f"Сегментов после постобработки: {len(segments)}")

    corrected_segments = correct_all_segments(segments)
    transcript_text = build_transcript_text(corrected_segments)
    print(f"Транскрипт: {len(transcript_text)} символов")

    plain_text = " ".join(s["text"] for s in corrected_segments)
    protocol = extract_meeting_protocol(plain_text)
    return transcript_text, protocol, meeting_id


def save_transcript_to_txt(transcript_text: str, meeting_id: str) -> str:
    """Сохраняет собранный транскрипт — отдельным файлом от протокола, рядом
    с ноутбуком (текущая рабочая директория)."""
    out_path = f"transcript_{meeting_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(transcript_text + "\n")
    print(f"Сохранено: {out_path}")
    return out_path


def save_protocol_to_txt(protocol: ProtocolOutput, meeting_id: str) -> str:
    """Сохраняет только протокол (JSON) — отдельным файлом от транскрипта."""
    out_path = f"protocol_{meeting_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(protocol.model_dump(), ensure_ascii=False, indent=2) + "\n")
    print(f"Сохранено: {out_path}")
    return out_path

# 8. Тестирование на реальном аудиофайле

Замените `AUDIO_LOCAL_PATH` на свой WAV-файл.


In [ ]:
AUDIO_LOCAL_PATH = "your_meeting_recording.wav"  # <-- впишите путь к своему аудиофайлу (WAV/OggOpus/MP3)


In [ ]:
transcript_text, protocol, meeting_id = run_meeting_pipeline(AUDIO_LOCAL_PATH)


In [ ]:
print("=" * 80)
print("ИСПРАВЛЕННЫЙ ТРАНСКРИПТ")
print("=" * 80)
print(transcript_text)

save_transcript_to_txt(transcript_text, meeting_id)


In [ ]:
print("=" * 80)
print("ПРОТОКОЛ")
print("=" * 80)
print(json.dumps(protocol.model_dump(), ensure_ascii=False, indent=2))

save_protocol_to_txt(protocol, meeting_id)


# 9. Результаты и анализ

## Что мы достигли

Полный цикл «аудио → протокол» на трёх сервисах Yandex Cloud:
- аудио загружено в Object Storage;
- SpeechKit вернул сегменты речи с таймкодами;
- YandexGPT батчами исправил ошибки ASR в сегментах (без изменения их количества и таймкодов), затем отдельным вызовом извлёк протокол из уже исправленного текста;
- транскрипт собран программно из исправленных сегментов и сохранён отдельно от протокола (`transcript_<id>.txt`, `protocol_<id>.txt`).

## Как это работает

Реплики подписаны отметкой времени (`[MM:SS]`), а не номером канала и не именами. Изначально мы брали `channelTag` от SpeechKit, но на практике оказалось, что запись с одного микрофона, сведённая в стерео, часто распознаётся SpeechKit на оба канала одинаково — получались два дублирующих друг друга набора реплик. Поэтому мы читаем текст только с одного канала (`channelTag == "0"`, см. `fetch_recognition` в разделе 5) и ориентируемся по времени, а не по каналу. Акустическая диаризация SpeechKit (`speakerLabeling`) тоже не помогла бы: она работает только для одноканальной записи и максимум на двух дикторов (см. раздел 5), а угадывание имён по одному тексту без голоса ненадёжно. Если нужны реальные имена, правильный путь — акустическая диаризация специализированным сервисом; этот кукбук её не показывает.

**Таймкоды никогда не проходят через текст, который генерирует LLM.** Первая версия кукбука встраивала метки `[MM:SS]` прямо в текст и просила модель "не трогать метки" при коррекции ASR — на части реальных записей модель эту инструкцию нарушала и стирала почти все метки, оставляя один `[00:00]` на весь транскрипт. Это не редкий баг конкретного промпта, а системная ненадёжность: просить LLM дословно пронести произвольный токен через переписывание текста — плохо гарантируемая вещь, и чем длиннее вход, тем чаще это ломается. Финальная архитектура следует стандартному паттерну ASR-пайплайнов (тот же принцип, что у forced-alignment инструментов вроде WhisperX): таймкод — это метаданные сегмента в питоновской структуре, а не часть текста. `correct_all_segments` отправляет модели только голый текст (JSON-массив без меток), получает обратно исправленный текст того же порядка и размера, и присоединяет к нему исходный `start_ms`/`end_ms` по индексу — программно, без парсинга из ответа модели. `build_transcript_text` затем собирает читаемый текст с редкими метками (раз в абзац, `PARAGRAPH_GAP_MS=20000`) — тоже чистая Python-функция, без обращения к LLM.

Длинные встречи не помещаются в один вызов коррекции — `batch_segments` режет список сегментов на батчи по суммарной длине текста, программно, без LLM-разметки границ (в отличие от прежнего подхода, где недорогой LLM-вызов размечал смысловые границы по номерам строк). Точная граница между батчами здесь не критична: батчи не делятся контекстом друг с другом, и раз сам текст сегмента при коррекции не меняется в длине радикально, случайная граница максимум разделит одну смысловую мысль на два соседних батча — в отличие от старой архитектуры, где неточная граница резала передаваемый текст посередине фразы.

Есть отдельный крайний случай: если в записи вообще нет пауз длиннее `MAX_PAUSE_MS_TO_MERGE`, вся запись (или её большая часть) склеивается в ОДИН сегмент — на реальном тесте так произошло с 10591-символьной записью. `batch_segments` резать умеет только между сегментами, а не внутри одного, поэтому гигантский сегмент раньше уходил в LLM целиком, упирался в `max_output_tokens`, и коррекция ASR обрывалась (fallback возвращал исходный текст без исправлений — пайплайн не падал, но текст оставался неисправленным). `_split_oversized_segment` дробит такой сегмент на под-сегменты по границам слов ещё до батчинга — все они наследуют таймкод родителя, точность которого здесь и так не выше (внутри одного склеенного блока нет собственных пауз, которые можно было бы использовать).

## Ключевые параметры

- `temperature=0.2` — консервативная настройка для задачи, где важна точность (коррекция чисел, дат), а не творческое разнообразие.
- `BATCH_SIZE_CHARS=6000` — суммарная длина текста сегментов в одном батче коррекции.
- `PARAGRAPH_GAP_MS=20000` — пауза между сегментами, после которой начинается новый абзац с новой меткой времени.
- `MAX_PAUSE_MS_TO_MERGE=8000` — пауза короче — сегменты SpeechKit склеиваются в один при постобработке.
- `GPT_MODEL=yandexgpt-5-lite` — выбрана по цене/скорости для учебного примера; для более сложных доменов сравните с `yandexgpt-5-pro` на своих записях.

## Ограничения учебной версии

- Извлечение протокола обрезает вход до 20 000 символов (`plain_text[:20000]`) — на очень длинных встречах (многочасовых) текст для протокола может не поместиться целиком. В проде этот шаг стоит развести на отдельные вызовы (промежуточные сводки по частям встречи + один финальный вызов на консолидацию).
- Реплики подписаны только отметкой времени, а не именами участников — если вам нужны реальные имена в протоколе, добавьте акустическую диаризацию (специализированный сервис) до этого пайплайна.
- Мы читаем только `channelTag == "0"` — если ваша запись действительно многоканальная (например, честный конференц-микшер с раздельными каналами на человека), эта эвристика отбросит второй канал вместе с реальным голосом. Проверьте на своей записи, дублируют ли каналы друг друга, прежде чем полагаться на это упрощение.
- Оценивайте качество коррекции на своих записях: возьмите 10-20 реальных фрагментов, аннотируйте эталонный текст руками и сравните с результатом модели, прежде чем фиксировать выбор модели в проде.

## Куда двигаться дальше

- Разнести старт распознавания и опрос операции по отдельным вызовам (Cloud Functions + очередь), если сервис асинхронный и без общего процесса ожидания.
- Добавить QA-проход, который проверяет пункты протокола на соответствие транскрипту.


# 10. Полезные ссылки

- [SpeechKit STT v3, асинхронное распознавание](https://aistudio.yandex.ru/docs/ru/speechkit/stt/api/transcribation-api-v3)
- [Поддерживаемые форматы аудио](https://aistudio.yandex.ru/docs/ru/speechkit/formats)
- [Доступные генеративные модели](https://aistudio.yandex.ru/docs/ru/ai-studio/concepts/generation/models)
- [Object Storage (S3-совместимое API)](https://yandex.cloud/ru/docs/storage/s3/)
